# 04 · LightGBM baseline, tuning, and which tables matter

Cross-validated LightGBM on the cleaned features, then a table-by-table ablation and a hyper-parameter search.

**Needs:** `features2.parquet` (02), `features3.parquet` (03). **Writes:** `oof_*.npy` (out-of-fold predictions), `search.json` (used by notebook 06). **Time:** about 1 hour (the random search dominates; lower `N_TRIALS` to go faster).

How to read the scores: every number is ROC-AUC from 5-fold stratified cross-validation (`random_state=0`), i.e. each patient is scored by a model that never saw them.

Findings: a first LightGBM gave 0.748; shallower trees gave 0.7514; the ablation showed labs (-0.0052 AUC if removed), discharge planning (-0.0034) and prior utilization (-0.0027) carry the signal, while vitals and medications add about 0.0004 each and unit transfers and procedures add nothing.

**Setup.** Install the libraries once (uncomment), then set the two paths in the next cell.

In [ ]:
# %pip install -q lightgbm xgboost scikit-learn pandas pyarrow scipy

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ---- EDIT THESE TWO PATHS --------------------------------------------------
DATA = os.environ.get('KAGGLE_DATA', 'data')   # folder holding the unzipped competition CSVs (train.csv, test.csv, patients.csv, ...)
WORK = os.environ.get('KAGGLE_WORK', 'work')   # folder where features, predictions and submissions are saved (shared by all notebooks)
# -----------------------------------------------------------------------------
Path(WORK).mkdir(parents=True, exist_ok=True)
print('DATA ->', Path(DATA).resolve()); print('WORK ->', Path(WORK).resolve())

### 4.1 First baseline (default-ish LightGBM) with feature importance

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, sys
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
D=DATA
X=pd.read_parquet(f'{WORK}/features.parquet')
tr=pd.read_csv(f'{D}/train.csv'); te=pd.read_csv(f'{D}/test.csv')
print('year dist train vs test'); 
for n,d in [('tr',tr),('te',te)]: print(n, X.set_index('encounter_id').loc[d.encounter_id].admit_year.value_counts(normalize=True).sort_index().round(3).to_dict())
cats=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
for c in cats: X[c]=X[c].astype('category')
T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values; feats=[c for c in T.columns if c not in ('encounter_id','readmit_30d')]
print(len(feats),'features; cats',len(cats))
p=dict(objective='binary',learning_rate=0.03,num_leaves=31,min_child_samples=100,feature_fraction=0.4,bagging_fraction=0.8,bagging_freq=1,lambda_l2=10,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
oof=np.zeros(len(T)); imp=pd.Series(0.,index=feats)
for f,(a,b) in enumerate(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)):
    m=lgb.train(p,lgb.Dataset(T.loc[a,feats],y[a]),3000,valid_sets=[lgb.Dataset(T.loc[b,feats],y[b])],callbacks=[lgb.early_stopping(150,verbose=False)])
    oof[b]=m.predict(T.loc[b,feats],num_iteration=m.best_iteration); imp+=pd.Series(m.feature_importance('gain'),index=feats)
    print(f,m.best_iteration,roc_auc_score(y[b],oof[b]),flush=True)
print('OOF AUC',roc_auc_score(y,oof))
imp=(imp/imp.sum()).sort_values(ascending=False); print(imp.head(40).round(4)); imp.to_csv(f'{WORK}/imp1.csv')
print('n feats with ~0 gain',(imp<1e-5).sum())
np.save(f'{WORK}/oof1.npy',oof)

### 4.2 A reusable cross-validation function
`cv(feature_file, tag, params, drop_prefixes)` runs 5-fold LightGBM with early stopping and saves the out-of-fold predictions as `oof_<tag>.npy`.

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, sys, json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def cv(feat_file, tag, pj, drop_prefix=()):
    drop_prefix = tuple(drop_prefix)
    D=DATA
    X=pd.read_parquet(feat_file); tr=pd.read_csv(f'{D}/train.csv')
    cats=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    for c in cats: X[c]=X[c].astype('category')
    T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values
    feats=[c for c in T.columns if c not in ('encounter_id','readmit_30d') and not c.startswith(drop_prefix)]
    p=dict(objective='binary',learning_rate=0.03,num_leaves=31,min_child_samples=100,feature_fraction=0.4,bagging_fraction=0.8,bagging_freq=1,lambda_l2=10,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
    p.update(pj)
    oof=np.zeros(len(T)); its=[]
    for f,(a,b) in enumerate(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)):
        m=lgb.train(p,lgb.Dataset(T.loc[a,feats],y[a]),4000,valid_sets=[lgb.Dataset(T.loc[b,feats],y[b])],callbacks=[lgb.early_stopping(200,verbose=False)])
        oof[b]=m.predict(T.loc[b,feats],num_iteration=m.best_iteration); its.append(m.best_iteration)
    print(tag,len(feats),'feats | OOF AUC %.5f'%roc_auc_score(y,oof),'| iters',its,flush=True)
    np.save(f'{WORK}/oof_{tag}.npy',oof)

In [ ]:
DROP = ("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
F2, F3 = f'{WORK}/features2.parquet', f'{WORK}/features3.parquet'
cv(F2, 'f2_base',    {})
cv(F2, 'f2_shallow', {"num_leaves":7,"min_child_samples":200,"feature_fraction":0.3,"lambda_l2":20,"learning_rate":0.04})
cv(F2, 'f2_d3',      {"num_leaves":4,"max_depth":2,"min_child_samples":300,"feature_fraction":0.3,"lambda_l2":30,"learning_rate":0.05})
cv(F2, 'f2_l5',      {"num_leaves":5,"min_child_samples":300,"feature_fraction":0.2,"lambda_l2":50,"learning_rate":0.05,"max_bin":63})

### 4.3 Table-by-table ablation (drop one group of features, see how much AUC is lost)

In [ ]:
P = {"num_leaves":5,"min_child_samples":300,"feature_fraction":0.2,"lambda_l2":50,"learning_rate":0.06,"max_bin":63}
cv(F2, 'ref', P)
ABLATE = {
  'no_vit':      ["vit_"],
  'no_med':      ["med_","mc_","mcont_","drug_","n_med","no_meds","meds_per"],
  'no_transfer': ["hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after"],
  'no_proc':     ["proc_","n_proc"],
  'no_dp':       ["dp_"],
  'no_pu':       ["pu_"],
  'no_lab':      ["lab_","egfr","bun_cr","cr_ratio","no_labs","labs_per"],
}
for tag, prefixes in ABLATE.items():
    cv(F2, tag, P, prefixes)

### 4.4 Old vs new features, same model (does 3.1 help?)

In [ ]:
cv(F2, 'old_feats', P, DROP)
cv(F3, 'new_feats', P, DROP)

### 4.5 Feature selection done *inside* each fold (so the score stays honest)

In [ ]:
import pandas as pd, numpy as np, lightgbm as lgb, sys, json
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
D=DATA; drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
X=pd.read_parquet(f'{WORK}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv')
for c in [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']: X[c]=X[c].astype('category')
T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values; feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
p=dict(objective='binary',learning_rate=0.06,num_leaves=5,min_child_samples=300,feature_fraction=0.2,bagging_fraction=0.8,bagging_freq=1,lambda_l2=50,max_bin=63,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
Ks=[100,200,300]; oof={k:np.zeros(len(T)) for k in Ks}
for f,(a,b) in enumerate(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)):
    m0=lgb.train(p,lgb.Dataset(T.loc[a,feats],y[a]),350)     # fixed rounds, selection from training part only
    g=pd.Series(m0.feature_importance('gain'),index=feats).sort_values(ascending=False)
    for k in Ks:
        fs=list(g.index[:k]); m=lgb.train(p,lgb.Dataset(T.loc[a,fs],y[a]),4000,valid_sets=[lgb.Dataset(T.loc[b,fs],y[b])],callbacks=[lgb.early_stopping(200,verbose=False)])
        oof[k][b]=m.predict(T.loc[b,fs],num_iteration=m.best_iteration)
    print('fold',f,{k:round(roc_auc_score(y[b],oof[k][b]),4) for k in Ks},flush=True)
    if f==0: print('top new-feature gains:',[i for i in g.index[:40] if i.startswith(('lx_','vx_','sev_','kidney','n_critical','unstable'))][:15])
print({k:round(roc_auc_score(y,v),5) for k,v in oof.items()})

### 4.6 Random hyper-parameter search
We planned 12 trials and stopped after 6 because nothing beat trial 0 (0.7514); all trials sat between 0.749 and 0.751. Each trial takes about 4 minutes on 2 cores.

In [ ]:
N_TRIALS = 6   # we ran 6 of a planned 12
import pandas as pd, numpy as np, lightgbm as lgb, json, time
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
D=DATA; drop=("hrs_","visits_","n_transfers","first_unit","last_unit","icu_","had_ed","ward_after","proc_","n_proc")
X=pd.read_parquet(f'{WORK}/features3.parquet'); tr=pd.read_csv(f'{D}/train.csv')
for c in [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and c!='encounter_id']: X[c]=X[c].astype('category')
T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values; feats=[c for c in T.columns if c not in('encounter_id','readmit_30d') and not c.startswith(drop)]
folds=list(StratifiedKFold(5,shuffle=True,random_state=0).split(T,y)); rs=np.random.RandomState(11); out=[]; t0=time.time()
base=dict(objective='binary',learning_rate=0.06,bagging_freq=1,verbose=-1,n_jobs=2,cat_smooth=50,max_cat_to_onehot=8)
for trial in range(N_TRIALS):
    q=dict(num_leaves=int(rs.choice([3,4,5,7,10,15])),min_child_samples=int(rs.choice([100,200,300,500,800])),feature_fraction=float(rs.choice([0.1,0.15,0.2,0.3,0.45])),
           bagging_fraction=float(rs.choice([0.6,0.8,1.0])),lambda_l2=float(rs.choice([5,20,50,100,200])),lambda_l1=float(rs.choice([0,1,5])),
           min_split_gain=float(rs.choice([0,0.5,2])),max_bin=int(rs.choice([31,63,127])),extra_trees=bool(rs.choice([False,True])),path_smooth=float(rs.choice([0,10,50])))
    p={**base,**q}; oof=np.zeros(len(T)); its=[]
    for a,b in folds:
        m=lgb.train(p,lgb.Dataset(T.loc[a,feats],y[a]),5000,valid_sets=[lgb.Dataset(T.loc[b,feats],y[b])],callbacks=[lgb.early_stopping(200,verbose=False)])
        oof[b]=m.predict(T.loc[b,feats],num_iteration=m.best_iteration); its.append(m.best_iteration)
    auc=roc_auc_score(y,oof); out.append((auc,q,its)); print(trial,'%.5f'%auc,q,'t=%.0f'%(time.time()-t0),flush=True)
    json.dump(sorted(out,key=lambda r:-r[0]),open(f'{WORK}/search.json','w'))
print('BEST',sorted(out,key=lambda r:-r[0])[:3])

### 4.7 Plain logistic regression for comparison (0.7446 on `features2`)

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
X=pd.read_parquet(f'{WORK}/features2.parquet'); tr=pd.read_csv(f'{DATA}/train.csv')
T=X.merge(tr,on='encounter_id'); y=T.readmit_30d.values
cat=[c for c in T.columns if not pd.api.types.is_numeric_dtype(T[c]) and c!='encounter_id']
num=[c for c in T.columns if c not in cat+['encounter_id','readmit_30d']]
num=[c for c in num if T[c].notna().sum()>0 and T[c].nunique()>1]
N=T[num].copy(); miss=N.columns[N.isna().mean()>0.01]
M=N[miss].isna().astype(int).add_suffix('_na')
N=N.apply(lambda s: s.clip(s.quantile(.005),s.quantile(.995))); N=N.fillna(N.median())
C=pd.get_dummies(T[cat].fillna('NA'),dtype=float)
Z=pd.concat([N,M,C],axis=1).astype(float)
print(Z.shape)
oof=np.zeros(len(T))
for a,b in StratifiedKFold(5,shuffle=True,random_state=0).split(Z,y):
    sc=StandardScaler().fit(Z.iloc[a]); m=LogisticRegression(C=0.02,max_iter=500).fit(sc.transform(Z.iloc[a]),y[a])
    oof[b]=m.predict_proba(sc.transform(Z.iloc[b]))[:,1]
print('LR OOF AUC',roc_auc_score(y,oof)); np.save(f'{WORK}/oof_lr.npy',oof)